<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/04-pronostica-sin-hacer-trampa/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 1.4 · Pronostica sin hacer trampa

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Qué es la **fuga de datos**, una de las trampas más comunes en Machine Learning.
- Usar solo lo que se sabe **antes** de predecir.
- Separar entrenamiento y prueba **por fechas**.
- Comparar un modelo con reglas simples que podrías hacer en Excel.

## 📊 En Excel lo harías así

Te piden cuánto comprará cada cliente el próximo mes. Lo más natural:

1. Armas una fila por cliente con sus últimos meses.
2. Sacas un `=PROMEDIO()` (`AVERAGE`) de los últimos 3 o 6 meses.
3. Ese es tu pronóstico.

No es una mala idea. De hecho, hoy va a ser muy difícil ganarle.

## 🐍 En Python

### Paso 1: la trampa

En el episodio 1.3, el modelo se equivocaba en unos S/ 700. Pero usaba las líneas de pedido del mes, y esas no se conocen hasta que el mes termina. Si ya sabes cuántas líneas pidió un cliente en julio, julio ya pasó.

Eso se llama **fuga de datos**: el modelo usa información que no tendrías al momento de predecir. En las pruebas se ve estupendo, y en la vida real no se puede usar.

> ⚠️ Antes de usar una columna, pregúntate: **¿la tendría en la mano el día que voy a predecir?**

### Paso 2: una tabla sin trampa

Esta función lista arma una fila por cliente y mes, con cero si ese mes no compró. Agrega lo que **sí** se sabe antes de empezar el mes: `mes_anterior`, `promedio_3` y `promedio_6`.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def tabla_de_pronostico(ventas, clientes):
    """Una fila por cliente y mes, con cero si no compró, y lo que se sabía antes de ese mes."""
    meses = ventas["fecha"].str[:7].rename("mes")
    todos = pd.MultiIndex.from_product(
        [clientes["id_cliente"], sorted(meses.unique())], names=["id_cliente", "mes"]
    )
    compras = ventas.groupby(["id_cliente", meses])["total"].sum()
    tabla = compras.reindex(todos, fill_value=0).reset_index()

    por_cliente = tabla.groupby("id_cliente")["total"]
    tabla["mes_anterior"] = por_cliente.shift(1)
    tabla["promedio_3"] = por_cliente.transform(lambda s: s.shift(1).rolling(3).mean())
    tabla["promedio_6"] = por_cliente.transform(lambda s: s.shift(1).rolling(6).mean())
    tabla["numero_mes"] = tabla["mes"].str[5:].astype(int)

    columnas = ["id_cliente", "tipo", "ciudad", "id_vendedor", "fecha_alta"]
    tabla = pd.merge(tabla, clientes[columnas], on="id_cliente", how="left")
    tabla = tabla.dropna(subset=["promedio_6"])
    return tabla[tabla["mes"] + "-01" >= tabla["fecha_alta"]].drop(columns="fecha_alta")

In [3]:
ventas = pd.read_csv(DATOS + "ventas.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")

tabla = tabla_de_pronostico(ventas, clientes)

print(tabla.shape)
tabla[["id_cliente", "mes", "total", "mes_anterior", "promedio_3", "promedio_6"]].head()

(5268, 10)


,id_cliente,mes,total,mes_anterior,promedio_3,promedio_6
6,C0001,2024-07,477.8,92.0,472.333333,327.800000
7,C0001,2024-08,344.0,477.8,526.266667,365.300000
8,C0001,2024-09,538.0,344.0,304.600000,373.133333
9,C0001,2024-10,1388.0,538.0,453.266667,462.800000
10,C0001,2024-11,310.0,1388.0,756.666667,641.466667


Fíjate en la primera fila: el `total` de julio de 2024 no entra en sus promedios, que miran solo hacia atrás. La tabla empieza en julio porque el promedio necesita 6 meses previos.

### Paso 3: separar por fechas

En el episodio 1.3 separamos al azar. Para pronosticar, eso sería otra trampa: entrenarías con diciembre para predecir octubre. La vida real solo va hacia adelante, así que **entrenamos con el pasado y probamos con el futuro**. Las fechas son texto en formato `aaaa-mm`, y por eso se pueden comparar con `<`. El promedio de octubre usa julio a septiembre, que también están en la prueba: no es trampa, porque cada mes se pronostica la víspera.

In [4]:
entreno = tabla[tabla["mes"] < "2025-07"]
prueba = tabla[tabla["mes"] >= "2025-07"]

print(len(entreno), len(prueba))

3475 1793


### Paso 4: las líneas base

Sin ningún modelo, solo con reglas, ¿cuánto nos equivocamos de julio a diciembre de 2025? Como antes, medimos con el MAE.

Para la primera regla usamos `.mean()` del entrenamiento, y así todos reciben el mismo número.

In [5]:
promedio_general = entreno["total"].mean()

print((prueba["total"] - promedio_general).abs().mean().round())
print(round(mean_absolute_error(prueba["total"], prueba["promedio_3"])))
print(round(mean_absolute_error(prueba["total"], prueba["promedio_6"])))

1555.0
889
855


Mirar la historia de cada cliente baja mucho el error: de unos S/ 1,555 a S/ 855. Y todo esto se puede hacer en Excel.

### Paso 5: ahora sí, un modelo

Entrenamos una recta con lo único que se sabe antes: el promedio de 6 meses.

In [6]:
modelo = LinearRegression()
modelo.fit(entreno[["promedio_6"]], entreno["total"])

prediccion = modelo.predict(prueba[["promedio_6"]])

print(modelo.coef_.round(2))
print(modelo.intercept_.round(1))
print(round(mean_absolute_error(prueba["total"], prediccion)))

[0.9]
235.9
885


**El modelo no le ganó a la regla.** Se equivoca en unos S/ 885, y el simple promedio de 6 meses, en S/ 855.

Lejos de ser un fracaso, es la lección más valiosa de hoy: si no hubiéramos medido la línea base, habríamos celebrado un modelo que no aporta. Y fíjate en la trampa: con estas mismas filas, el modelo con líneas de pedido da unos S/ 693, mucho mejor que S/ 885. Por eso **un resultado demasiado bueno debe hacerte sospechar una fuga**.

¿Por qué pierde? Sobre todo porque a quien no compra le sigue pronosticando unos S/ 236: lo verás en el ejercicio 6. Y tampoco sabe de temporadas, algo que veremos en el próximo episodio.

## 🤖 Pídeselo a la IA

> *"Quiero predecir si un paciente volverá a su cita. Tengo estas columnas: edad, distrito, citas previas, hora de llegada y si pagó la consulta. ¿Cuáles serían fuga de datos? Explícame por qué."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Usar solo los meses anteriores | Columnas como `promedio_6`, que miran hacia atrás |
| Proyectar con el pasado y comparar con el futuro | `tabla[tabla["mes"] < "2025-07"]` y `tabla[tabla["mes"] >= "2025-07"]` |
| Promedio de los últimos meses como proyección | Línea base: `mean_absolute_error(prueba["total"], prueba["promedio_6"])` |
| | **Fuga de datos:** usar lo que no se sabría al predecir |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/04-pronostica-sin-hacer-trampa/ejercicio.ipynb) y busca las trampas. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/04-pronostica-sin-hacer-trampa/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0